In [1]:
import argparse
import cv2
import numpy as np
import onnx
import onnxruntime as ort
from pathlib import Path
import time
from challenge.tinyyolov2 import TinyYoloV2, TinyYoloV2_BNOpt
import torch
from typing import List
from challenge.utils.viz import num_to_class
from challenge.utils.yolo import nms, filter_boxes

## onnx conversion

In [5]:
ort.get_device()

'GPU'

In [1]:
from onnxruntime.quantization import quantize_dynamic, QuantType


In [6]:
qmodel = quantize_dynamic('../models/person_only_baseline/model_best_bnopt.onnx', '../models/person_only_baseline/qmodel.onnx', weight_type=QuantType.QUInt8)


In [ ]:
'''
python -m onnxruntime.quantization.preprocess --input mobilenetv2-7.onnx --output mobilenetv2-7-infer.onnx
python run.py --input_model mobilenetv2-7-infer.onnx --output_model mobilenetv2-7.quant.onnx 
'''

In [10]:
import onnx
from onnxconverter_common import float16

In [8]:
modelfp32_path = '../models/person_only_baseline/model_best_bnopt.onnx'
modelfp16_path = '../models/person_only_baseline/fp16model.onnx'

In [11]:
model = onnx.load(modelfp32_path)
model_fp16 = float16.convert_float_to_float16(model)
onnx.save(model_fp16, modelfp16_path)

c:\Users\hnamt\anaconda3\envs\mlenv2\lib\site-packages\onnxconverter_common\float16.py:53: UserWarning: the float32 number -3.049829544465865e-08 will be truncated to -1e-07
  warnings.warn("the float32 number {} will be truncated to {}".format(neg_max, -min_positive_val))
c:\Users\hnamt\anaconda3\envs\mlenv2\lib\site-packages\onnxconverter_common\float16.py:53: UserWarning: the float32 number -2.319551661855712e-08 will be truncated to -1e-07
  warnings.warn("the float32 number {} will be truncated to {}".format(neg_max, -min_positive_val))
c:\Users\hnamt\anaconda3\envs\mlenv2\lib\site-packages\onnxconverter_common\float16.py:43: UserWarning: the float32 number 3.080014465695058e-08 will be truncated to 1e-07
  warnings.warn("the float32 number {} will be truncated to {}".format(pos_min, min_positive_val))
c:\Users\hnamt\anaconda3\envs\mlenv2\lib\site-packages\onnxconverter_common\float16.py:53: UserWarning: the float32 number -4.195497993464414e-08 will be truncated to -1e-07
  warni

## trt

In [12]:
import tensorrt as trt

In [ ]:
import tensorrt as trt

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
trt_runtime = trt.Runtime(TRT_LOGGER)
def build_engine(onnx_path, shape = [1,224,224,3]):
    """
    This is the function to create the TensorRT engine
    Args:
        onnx_path : Path to onnx_file. 
        shape : Shape of the input of the ONNX file. 
    """
    with trt.Builder(TRT_LOGGER) as builder, builder.create_network(1) as network, builder.create_builder_config() as config, trt.OnnxParser(network, TRT_LOGGER) as parser:
       config.max_workspace_size = (256 << 20)
       with open(onnx_path, 'rb') as model:
           parser.parse(model.read())
       network.get_input(0).shape = shape
       engine = builder.build_engine(network, config)
       return engine

def save_engine(engine, file_name):
   buf = engine.serialize()
   with open(file_name, 'wb') as f:
       f.write(buf)
def load_engine(trt_runtime, plan_path):
   with open(plan_path, 'rb') as f:
       engine_data = f.read()
   engine = trt_runtime.deserialize_cuda_engine(engine_data)
   return engine

In [ ]:
import engine as eng
import argparse
from onnx import ModelProto
import tensorrt as trt 
 
 engine_name = “resnet50.plan”
 onnx_path = "/path/to/onnx/result/file/"
 batch_size = 1 
 
 model = ModelProto()
 with open(onnx_path, "rb") as f:
    model.ParseFromString(f.read())
 
 d0 = model.graph.input[0].type.tensor_type.shape.dim[1].dim_value
 d1 = model.graph.input[0].type.tensor_type.shape.dim[2].dim_value
 d2 = model.graph.input[0].type.tensor_type.shape.dim[3].dim_value
 shape = [batch_size , d0, d1 ,d2]
 engine = eng.build_engine(onnx_path, shape= shape)
 eng.save_engine(engine, engine_name) 

In [2]:
from torch2trt import torch2trt

ModuleNotFoundError: No module named 'torch2trt'